# Лабораторная работа № 3
## Обработка исключений. Модуль OS. Виртуальные окружения

**Среда выполнения:** Google Colab + GitLab

**Прикладная задача:** «Пакетный импорт JSON-документов» — конвейер, который рекурсивно читает JSON-файлы из каталога, валидирует их по обязательным полям, изолирует ошибки по каждому файлу, пишет логи и формирует сводный отчёт.

**Автор:** <ФИО>

**Группа:** <номер группы>

**Дата:** <дата>

---
## Часть I. Теория

### 1. Исключения

**Исключение** — объект, сигнализирующий об ошибке во время выполнения. Если его не обработать, программа завершится аварийно и покажет traceback.

Синтаксис:

```python
try:
    # код, в котором может возникнуть ошибка
except SomeError as exc:
    # обработка
else:
    # выполняется, если ошибки не было
finally:
    # выполняется всегда
```

Порядок выполнения:

1. Интерпретатор входит в `try` и выполняет код.
2. Если исключения не было — управление уходит в `else`, минуя `except`.
3. Если возникло исключение — ищется подходящий `except`. Первый совпавший выполняется.
4. `finally` выполняется всегда: при ошибке, без ошибки, при `return` и даже при `raise`.

### 1.1. Иерархия исключений

Все исключения наследуются от `BaseException`. Основные ветки:

| Класс | Назначение |
|---|---|
| `SystemExit` | `sys.exit()` |
| `KeyboardInterrupt` | Ctrl+C |
| `GeneratorExit` | закрытие генератора |
| `Exception` | база для «обычных» ошибок |

Примеры внутри `Exception`:

| Ветка | Потомки | Пример |
|---|---|---|
| `ArithmeticError` | `ZeroDivisionError`, `OverflowError` | `1 / 0` |
| `LookupError` | `IndexError`, `KeyError` | `[1, 2][5]` |
| `OSError` | `FileNotFoundError`, `PermissionError` | `open("нет.txt")` |
| `ValueError` | — | `int("abc")` |
| `TypeError` | — | `"5" + 5` |
| `AttributeError` | — | `(5).foo` |

**Правило:** перехватывайте конкретные типы, а не `except:` и не `except Exception:` без нужды. Голый `except:` ловит в том числе `KeyboardInterrupt`.

### 1.2. Собственные исключения

```python
class AppError(Exception):
    """Базовая ошибка приложения."""

class ValidationError(AppError):
    """Данные не прошли валидацию."""
```

Наследуйте от `Exception`. Имя класса заканчивается на `Error`. Сообщение должно объяснять, что случилось.

### 1.3. LBYL и EAFP

- **LBYL** (Look Before You Leap) — сначала проверить условие, потом действовать.
- **EAFP** (Easier to Ask Forgiveness than Permission) — действовать, а ошибку обрабатывать по факту.

В Python идиоматичен EAFP: между проверкой и действием состояние может измениться (файл исчезнет, объект перестанет существовать).

### 1.4. Лучшие практики

- Ловить конкретное исключение, а не всё подряд.
- Не использовать исключения для управления потоком.
- Сохранять исходный traceback при оборачивании: `raise NewError(...) from exc`.
- Не проглатывать исключения молча (`except: pass`).
- Собственные исключения — под предметную область, а не «на всякий случай».

---
### 2. Работа с файлами

Режимы открытия:

| Режим | Назначение | Создаёт? | Перезаписывает? |
|---|---|---|---|
| `'r'` | Чтение | Нет | — |
| `'w'` | Запись | Да | Да |
| `'a'` | Дозапись | Да | Нет |
| `'x'` | Создание | Да (ошибка, если есть) | — |
| `'b'` | Бинарный режим | — | — |

Правила:

- Всегда указывать `encoding="utf-8"` для текстовых файлов.
- Всегда использовать `with open(...) as f:`.
- Для больших файлов — построчное чтение: `for line in f:`.

In [1]:
# Запись и чтение текстового файла
with open("demo.txt", "w", encoding="utf-8") as f:
    f.write("Строка 1\n")
    f.write("Строка 2\n")

with open("demo.txt", "r", encoding="utf-8") as f:
    content = f.read()
print(content)

Строка 1
Строка 2



### 2.1. JSON

```python
import json

with open("data.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

with open("data.json", "r", encoding="utf-8") as f:
    loaded = json.load(f)
```

`ensure_ascii=False` — чтобы кириллица не превращалась в `\uXXXX`. `json.loads` / `json.dumps` — для строк; `json.load` / `json.dump` — для файлов. `json.JSONDecodeError` — исключение при битом JSON.

In [2]:
import json

data = {"name": "Иван", "courses": ["Python", "Алгоритмы"]}

with open("data.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

with open("data.json", "r", encoding="utf-8") as f:
    loaded = json.load(f)

print(loaded)
print(type(loaded["courses"]))

{'name': 'Иван', 'courses': ['Python', 'Алгоритмы']}
<class 'list'>


---
### 3. Модули `os`, `os.path`, `pathlib`

**`os`** — прослойка между Python и операционной системой: `getcwd`, `chdir`, `listdir`, `mkdir`, `makedirs`, `remove`, `rename`, `environ`.

**`os.path`** — работа с путями как со строками: `join`, `basename`, `dirname`, `split`, `splitext`, `exists`, `abspath`, `getsize`.

**`pathlib`** — современный объектно-ориентированный подход:

```python
from pathlib import Path

p = Path("data") / "orders.txt"
p.write_text("...", encoding="utf-8")
text = p.read_text(encoding="utf-8")
for f in Path("data").rglob("*.json"):
    ...
```

**Правило:** в новом коде — `pathlib`. `os.path` — для унаследованного кода.

In [3]:
from pathlib import Path

p = Path("demo") / "example.txt"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("Привет, pathlib!", encoding="utf-8")

print("Путь:", p)
print("name:", p.name, "| stem:", p.stem, "| suffix:", p.suffix)
print("Содержимое:", p.read_text(encoding="utf-8"))
print("Размер:", p.stat().st_size, "байт")

Путь: demo/example.txt
name: example.txt | stem: example | suffix: .txt
Содержимое: Привет, pathlib!
Размер: 22 байт


---
### 4. Модули и пакеты

**Модуль** — файл `.py`. **Пакет** — каталог с модулями и файлом `__init__.py`.

Варианты импорта:

```python
import math                      # math.sqrt(4)
from math import sqrt            # sqrt(4)
from math import sqrt as sq      # sq(4)
from math import *               # плохо: засоряет пространство имён
```

**Идиома `if __name__ == "__main__":`** отделяет код для запуска от кода для импорта.

In [4]:
%%writefile jsonutils.py
"""Утилиты для работы с JSON."""
import json
from pathlib import Path


def read_json(path):
    """Читает JSON-файл и возвращает словарь."""
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_json(data, path):
    """Записывает словарь в JSON-файл с сохранением кириллицы."""
    Path(path).write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


if __name__ == "__main__":
    write_json({"hello": "мир"}, "demo_module.json")
    print(read_json("demo_module.json"))

Writing jsonutils.py


In [5]:
import jsonutils

jsonutils.write_json({"a": 1, "b": "текст"}, "module_test.json")
print(jsonutils.read_json("module_test.json"))

{'a': 1, 'b': 'текст'}


---
### 5. Виртуальные окружения, `pip`, `requirements.txt`, `.env`

**Виртуальное окружение (`venv`)** — изолированная песочница для зависимостей проекта. Решает конфликт версий между проектами.

```bash
python -m venv .venv
source .venv/bin/activate         # Linux / macOS
.venv\Scripts\activate            # Windows
pip install pandas
pip freeze > requirements.txt
```

**`requirements.txt`** — список зависимостей проекта.

**`.env` и `.gitignore`:**

```
# .env
API_KEY=super-secret-key

# .gitignore
.env
venv/
__pycache__/
```

Секреты — только в окружении, не в коде и не в Git.

In [32]:
!pip install python-dotenv --quiet

UsageError: Line magic function `%%writefile` not found.


In [33]:
%%writefile .env
SOURCE_DIR=inbox
OUTPUT_PATH=catalog.json
REQUIRED_KEYS=id,name

Overwriting .env


In [34]:
%%writefile .gitignore
.env
venv/
venv_*/
__pycache__/
*.pyc
*.log
.idea/
.vscode/
.ipynb_checkpoints/

Writing .gitignore


In [35]:
from dotenv import load_dotenv
import os

load_dotenv()
print("SOURCE_DIR:", os.getenv("SOURCE_DIR"))
print("OUTPUT_PATH:", os.getenv("OUTPUT_PATH"))
print("REQUIRED_KEYS:", os.getenv("REQUIRED_KEYS"))

SOURCE_DIR: inbox
OUTPUT_PATH: catalog.json
REQUIRED_KEYS: id,name


---
### 6. Логирование

`print` не подходит для продакшена: нет уровней, меток времени, нельзя направить в файл.

Уровни: `DEBUG < INFO < WARNING < ERROR < CRITICAL`.

Архитектура:

- **Logger** — точка входа (`logging.getLogger(__name__)`).
- **Handler** — куда писать (консоль, файл, syslog).
- **Formatter** — как писать (формат строки).

Ротация: `RotatingFileHandler`, `TimedRotatingFileHandler`.

Ленивое форматирование: `logger.info("User %s", user)`, а не f-строка.

In [8]:
import logging
from logging.handlers import RotatingFileHandler

logger = logging.getLogger("demo")
logger.setLevel(logging.DEBUG)

console = logging.StreamHandler()
console.setLevel(logging.INFO)
console.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))

file_handler = RotatingFileHandler(
    "demo.log", maxBytes=100_000, backupCount=3, encoding="utf-8"
)
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(
    logging.Formatter("%(asctime)s [%(levelname)s] %(name)s: %(message)s")
)

logger.addHandler(console)
logger.addHandler(file_handler)

logger.debug("Только в файл")
logger.info("И в консоль, и в файл")
logger.warning("Предупреждение")

print("\nСодержимое demo.log:")
print(open("demo.log", encoding="utf-8").read())

DEBUG:demo:Только в файл
INFO: И в консоль, и в файл
INFO:demo:И в консоль, и в файл



Содержимое demo.log:
2026-10-09 06:01:02,362 [DEBUG] demo: Только в файл
2026-10-09 06:01:02,363 [INFO] demo: И в консоль, и в файл
2026-10-09 06:01:02,365 [WARNING] demo: Предупреждение



---
## Часть II. Задания

### Задание 1. Функция vs процедура в обработке ошибок

1. Процедура `log_error(path: str, reason: str) -> None` — логирует ошибку через `logging.warning`.
2. Функция `parse_int_safe(value: str) -> int | None` — возвращает число или `None` при `ValueError`.

In [9]:
# Реализация
import logging

logger = logging.getLogger("task1")
logging.basicConfig(level=logging.INFO)


def log_error(path: str, reason: str) -> None:
    """Процедура: логирует ошибку и ничего не возвращает."""
    ...


def parse_int_safe(value: str) -> int | None:
    """Функция: возвращает int или None при ValueError."""
    ...

# Проверки
log_error("data.json", "не найден")
print(parse_int_safe("42"))
print(parse_int_safe("abc"))

None
None


### Задание 2. Все виды исключений

Для каждого выражения предскажите тип исключения, затем проверьте в try/except. Покажите `__mro__` и объясните, от кого наследуется.

```python
1 / 0
int("abc")
[1, 2][5]
{"a": 1}["b"]
"5" + 5
open("no_such_file.txt")
```

In [10]:
# Реализация
expressions = [
    ("1 / 0", lambda: 1 / 0),
    ("int('abc')", lambda: int("abc")),
    ("[1, 2][5]", lambda: [1, 2][5]),
    ("{'a': 1}['b']", lambda: {"a": 1}["b"]),
    ("'5' + 5", lambda: "5" + 5),
    ("open('no_such_file.txt')", lambda: open("no_such_file.txt")),
]

for label, func in expressions:
    try:
        func()
    except Exception as exc:
        print(f"{label:30s} -> {type(exc).__name__}")
        print("    MRO:", [c.__name__ for c in type(exc).__mro__])

1 / 0                          -> ZeroDivisionError
    MRO: ['ZeroDivisionError', 'ArithmeticError', 'Exception', 'BaseException', 'object']
int('abc')                     -> ValueError
    MRO: ['ValueError', 'Exception', 'BaseException', 'object']
[1, 2][5]                      -> IndexError
    MRO: ['IndexError', 'LookupError', 'Exception', 'BaseException', 'object']
{'a': 1}['b']                  -> KeyError
    MRO: ['KeyError', 'LookupError', 'Exception', 'BaseException', 'object']
'5' + 5                        -> TypeError
    MRO: ['TypeError', 'Exception', 'BaseException', 'object']
open('no_such_file.txt')       -> FileNotFoundError
    MRO: ['FileNotFoundError', 'OSError', 'Exception', 'BaseException', 'object']


### Задание 3. try / except / else / finally

Реализуйте `safe_divide(a, b) -> float | None`:
- в `try` — деление;
- в `except ZeroDivisionError` — лог и `None`;
- в `else` — сообщение об успехе;
- в `finally` — сообщение «операция завершена».

Продемонстрируйте: `(10, 2)`, `(10, 0)`, `(10, "x")`.

In [11]:
def safe_divide(a, b) -> float | None:
    ...

print(safe_divide(10, 2))
print(safe_divide(10, 0))
print(safe_divide(10, "x"))

None
None
None


### Задание 4. Своя иерархия исключений

Создайте `IngestError`, `ConfigError`, `DocumentError`. Продемонстрируйте, что `except IngestError` ловит обоих наследников.

In [12]:
class IngestError(Exception):
    """Базовая ошибка конвейера импорта."""


class ConfigError(IngestError):
    """Настройки невалидны."""


class DocumentError(IngestError):
    """Проблема с конкретным документом."""

    def __init__(self, path, reason: str) -> None:
        super().__init__(f"{path}: {reason}")
        self.path = path
        self.reason = reason


# Проверки
for exc in (ConfigError("нет каталога"), DocumentError("a.json", "битый JSON")):
    try:
        raise exc
    except IngestError as e:
        print(type(e).__name__, "->", e)

ConfigError -> нет каталога
DocumentError -> a.json: битый JSON


### Задание 5. LBYL vs EAFP

Реализуйте чтение файла двумя способами. Объясните, в каком случае EAFP надёжнее.

In [13]:
import os

path = "no_such_file.txt"

# LBYL
if os.path.exists(path):
    with open(path) as f:
        data = f.read()
else:
    print("LBYL: файла нет")

# EAFP
try:
    with open(path) as f:
        data = f.read()
except FileNotFoundError:
    print("EAFP: файла нет")

LBYL: файла нет
EAFP: файла нет


### Задание 6. Файловые режимы

Создайте файл в режиме `'w'`, допишите в режиме `'a'`, прочитайте. Объясните, что произойдёт при повторном открытии в `'w'`.

In [14]:
with open("test.txt", "w", encoding="utf-8") as f:
    f.write("Строка 1\nСтрока 2\nСтрока 3\n")

with open("test.txt", "a", encoding="utf-8") as f:
    f.write("Строка 4\n")

with open("test.txt", "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        print(i, line.rstrip())

1 Строка 1
2 Строка 2
3 Строка 3
4 Строка 4


### Задание 7. JSON: сериализация и десериализация

Сохраните словарь с кириллицей через `json.dump` с `ensure_ascii=False`, прочитайте обратно. Поймайте `json.JSONDecodeError` на битом JSON.

In [15]:
import json

data = {"город": "Москва", "население": 12_500_000, "районы": ["ЦАО", "САО"]}

with open("city.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

with open("city.json", "r", encoding="utf-8") as f:
    loaded = json.load(f)
print("Прочитано:", loaded)

try:
    json.loads('{"a": 1, "b": }')
except json.JSONDecodeError as exc:
    print("Ошибка JSON:", exc)

Прочитано: {'город': 'Москва', 'население': 12500000, 'районы': ['ЦАО', 'САО']}
Ошибка JSON: Expecting value: line 1 column 15 (char 14)


### Задание 8. Работа с `pathlib`

Создайте дерево каталогов, запишите два JSON-файла, обойдите все `.json` через `rglob`, выведите имя, размер и полный путь для каждого.

In [16]:
from pathlib import Path

Path("tree/raw").mkdir(parents=True, exist_ok=True)
Path("tree/processed").mkdir(parents=True, exist_ok=True)

Path("tree/raw/a.json").write_text('{"a": 1}', encoding="utf-8")
Path("tree/raw/b.json").write_text('{"b": 2}', encoding="utf-8")

for p in Path("tree").rglob("*.json"):
    print(f"{p.name:10s} {p.stat().st_size:>5} B   {p.absolute()}")

b.json         8 B   /content/tree/raw/b.json
a.json         8 B   /content/tree/raw/a.json


### Задание 9. Свой модуль

Создайте файл `jsonutils.py` с функциями `read_json` и `write_json`, блоком `if __name__ == "__main__"`. Импортируйте и используйте его в ноутбуке.

In [17]:
%%writefile jsonutils.py
"""Утилиты для работы с JSON."""
import json
from pathlib import Path


def read_json(path) -> dict:
    """Читает JSON-файл и возвращает словарь."""
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_json(data: dict, path) -> None:
    """Записывает словарь в JSON-файл."""
    Path(path).write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


if __name__ == "__main__":
    write_json({"hello": "мир"}, "demo_module.json")
    print(read_json("demo_module.json"))

Overwriting jsonutils.py


In [18]:
import jsonutils

jsonutils.write_json({"a": 1, "b": "текст"}, "module_test.json")
print(jsonutils.read_json("module_test.json"))

{'a': 1, 'b': 'текст'}


### Задание 10. Логирование с уровнями

Настройте логгер с двумя handler'ами (консоль INFO, файл DEBUG с ротацией). Выведите все пять уровней. Проверьте содержимое файла.

In [19]:
import logging
from logging.handlers import RotatingFileHandler

logger = logging.getLogger("task10")
logger.setLevel(logging.DEBUG)

console = logging.StreamHandler()
console.setLevel(logging.INFO)
console.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))

fh = RotatingFileHandler("task10.log", maxBytes=100_000, backupCount=3, encoding="utf-8")
fh.setLevel(logging.DEBUG)
fh.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(name)s: %(message)s"))

logger.addHandler(console)
logger.addHandler(fh)

logger.debug("DEBUG")
logger.info("INFO")
logger.warning("WARNING")
logger.error("ERROR")
logger.critical("CRITICAL")

print("\ntask10.log:")
print(open("task10.log", encoding="utf-8").read())

DEBUG:task10:DEBUG
INFO: INFO
INFO:task10:INFO
ERROR: ERROR
ERROR:task10:ERROR
CRITICAL: CRITICAL
CRITICAL:task10:CRITICAL



task10.log:
2026-10-09 06:01:02,587 [DEBUG] task10: DEBUG
2026-10-09 06:01:02,589 [INFO] task10: INFO
2026-10-09 06:01:02,591 [WARNING] task10: WARNING
2026-10-09 06:01:02,598 [ERROR] task10: ERROR
2026-10-09 06:01:02,601 [CRITICAL] task10: CRITICAL



---
### Инициализация данных для заданий 11–20

Запустите эту ячейку один раз — данные будут использоваться во всех последующих заданиях.

In [20]:
import shutil
from pathlib import Path

shutil.rmtree("inbox", ignore_errors=True)
Path("inbox/good").mkdir(parents=True)
Path("inbox/nested").mkdir(parents=True)

Path("inbox/good/a.json").write_text(
    '{"id": 1, "name": "Alice", "age": 30}', encoding="utf-8"
)
Path("inbox/good/b.json").write_text(
    '{"id": 2, "name": "Bob", "age": 25}', encoding="utf-8"
)
Path("inbox/nested/c.json").write_text(
    '{"id": 3, "name": "Carol", "age": 28}', encoding="utf-8"
)
Path("inbox/broken.json").write_text(
    '{"id": 4, "name": }', encoding="utf-8"
)
Path("inbox/no_keys.json").write_text(
    '{"foo": "bar"}', encoding="utf-8"
)
Path("inbox/not_object.json").write_text(
    '[1, 2, 3]', encoding="utf-8"
)

print(f"Всего файлов: {len(list(Path('inbox').rglob('*.json')))}")

Всего файлов: 6


### Задание 11. Настройки из окружения

Класс `Settings` с полями `source_dir`, `output_path`, `required_keys`. Метод `from_env()` читает окружение и валидирует.

In [21]:
import os
from pathlib import Path


class ConfigError(Exception):
    """Настройки невалидны."""


class Settings:
    def __init__(self, source_dir: Path, output_path: Path, required_keys: tuple[str, ...]) -> None:
        self.source_dir = source_dir
        self.output_path = output_path
        self.required_keys = required_keys

    @classmethod
    def from_env(cls) -> "Settings":
        source = Path(os.environ.get("SOURCE_DIR", "inbox"))
        output = Path(os.environ.get("OUTPUT_PATH", "catalog.json"))
        keys = tuple(k.strip() for k in os.environ.get("REQUIRED_KEYS", "id,name").split(",") if k.strip())

        if not source.exists():
            raise ConfigError(f"Каталог не найден: {source}")
        if not source.is_dir():
            raise ConfigError(f"Ожидался каталог: {source}")
        if not keys:
            raise ConfigError("REQUIRED_KEYS пустой")

        return cls(source, output, keys)


os.environ["SOURCE_DIR"] = "inbox"
os.environ["REQUIRED_KEYS"] = "id,name"
settings = Settings.from_env()
print(settings.source_dir, settings.output_path, settings.required_keys)

inbox catalog.json ('id', 'name')


### Задание 12. Обход каталога

Метод `Ingester.iter_files()` — рекурсивный обход через `Path.rglob`, отсортированный итератор.

In [22]:
class Ingester:
    def __init__(self, settings: Settings) -> None:
        self.settings = settings
        self.errors: list[str] = []

    def iter_files(self):
        """Все .json в source_dir, рекурсивно, отсортированные."""
        return iter(sorted(self.settings.source_dir.rglob("*.json")))


ing = Ingester(settings)
for p in ing.iter_files():
    print(p)

inbox/broken.json
inbox/good/a.json
inbox/good/b.json
inbox/nested/c.json
inbox/no_keys.json
inbox/not_object.json


### Задание 13. Чтение одного документа

Метод `load_document(path) -> dict` с полной валидацией. При проблеме — `DocumentError` с причиной. Ошибки `OSError` / `JSONDecodeError` оборачиваются через `raise ... from exc`.

In [23]:
import json


class DocumentError(IngestError_ := Exception):  # временная база
    def __init__(self, path: Path, reason: str) -> None:
        super().__init__(f"{path.name}: {reason}")
        self.path = path
        self.reason = reason


class IngesterWithLoad(Ingester):
    def load_document(self, path: Path) -> dict:
        try:
            text = path.read_text(encoding="utf-8")
        except OSError as exc:
            raise DocumentError(path, f"не читается ({exc.strerror})") from exc

        try:
            data = json.loads(text)
        except json.JSONDecodeError as exc:
            raise DocumentError(path, f"битый JSON ({exc.msg})") from exc

        if not isinstance(data, dict):
            raise DocumentError(path, f"ожидался объект, получено {type(data).__name__}")

        missing = [k for k in self.settings.required_keys if k not in data]
        if missing:
            raise DocumentError(path, f"нет полей: {', '.join(missing)}")

        return data


ing = IngesterWithLoad(settings)
for p in ing.iter_files():
    try:
        data = ing.load_document(p)
        print(f"OK    {p.name}: {data}")
    except DocumentError as exc:
        print(f"FAIL  {exc}")

FAIL  broken.json: битый JSON (Expecting value)
OK    a.json: {'id': 1, 'name': 'Alice', 'age': 30}
OK    b.json: {'id': 2, 'name': 'Bob', 'age': 25}
OK    c.json: {'id': 3, 'name': 'Carol', 'age': 28}
FAIL  no_keys.json: нет полей: id, name
FAIL  not_object.json: ожидался объект, получено list


### Задание 14. Конвейер с изоляцией ошибок

Метод `run() -> dict` — обходит файлы, изолирует ошибки, формирует отчёт.

In [24]:
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
logger = logging.getLogger("ingest")


class IngesterRun(IngesterWithLoad):
    def run(self) -> dict:
        files = list(self.iter_files())
        records: list[dict] = []
        skipped = 0

        for path in files:
            try:
                records.append(self.load_document(path))
            except DocumentError as exc:
                skipped += 1
                self.errors.append(str(exc))
                logger.warning("Пропущен %s", exc)
            except Exception:
                skipped += 1
                logger.exception("Неожиданная ошибка на %s", path)

        return {
            "source": str(self.settings.source_dir),
            "total_files": len(files),
            "imported": len(records),
            "skipped": skipped,
            "errors": self.errors,
            "records": records,
        }


report = IngesterRun(settings).run()
for k, v in report.items():
    if k != "records":
        print(f"{k}: {v}")
print("records:", report["records"])

source: inbox
total_files: 6
imported: 3
skipped: 3
errors: ['broken.json: битый JSON (Expecting value)', 'no_keys.json: нет полей: id, name', 'not_object.json: ожидался объект, получено list']
records: [{'id': 1, 'name': 'Alice', 'age': 30}, {'id': 2, 'name': 'Bob', 'age': 25}, {'id': 3, 'name': 'Carol', 'age': 28}]


### Задание 15. Сохранение отчёта

Метод `_save(report)` — создаёт родительский каталог, записывает JSON с `ensure_ascii=False`.

In [25]:
class IngesterFull(IngesterRun):
    def _save(self, report: dict) -> None:
        try:
            self.settings.output_path.parent.mkdir(parents=True, exist_ok=True)
            self.settings.output_path.write_text(
                json.dumps(report, ensure_ascii=False, indent=2),
                encoding="utf-8",
            )
        except OSError as exc:
            raise ConfigError(f"Не удалось записать {self.settings.output_path}: {exc}") from exc
        logger.info("Отчёт сохранён: %s", self.settings.output_path)

    def run(self) -> dict:
        report = super().run()
        self._save(report)
        return report


report = IngesterFull(settings).run()
print("Отчёт сохранён в", settings.output_path)

Отчёт сохранён в catalog.json


### Задание 16. Настройка логирования

Функция `setup_logging()` — консоль INFO, файл DEBUG с ротацией, единый формат.

In [26]:
import logging
from logging.handlers import RotatingFileHandler


def setup_logging() -> None:
    root = logging.getLogger()
    root.setLevel(logging.DEBUG)

    console = logging.StreamHandler()
    console.setLevel(logging.INFO)
    console.setFormatter(logging.Formatter("%(levelname)s %(name)s: %(message)s"))

    fh = RotatingFileHandler("ingest.log", maxBytes=100_000, backupCount=3, encoding="utf-8")
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(name)s: %(message)s"))

    root.addHandler(console)
    root.addHandler(fh)


setup_logging()
IngesterFull(settings).run()
print("\nСодержимое ingest.log:")
print(open("ingest.log", encoding="utf-8").read())

WARNING ingest: Пропущен broken.json: битый JSON (Expecting value)
WARNING ingest: Пропущен no_keys.json: нет полей: id, name
WARNING ingest: Пропущен not_object.json: ожидался объект, получено list
INFO:ingest:Отчёт сохранён: catalog.json
INFO ingest: Отчёт сохранён: catalog.json



Содержимое ingest.log:
2026-10-09 06:01:02,804 [WARNING] ingest: Пропущен broken.json: битый JSON (Expecting value)
2026-10-09 06:01:02,808 [WARNING] ingest: Пропущен no_keys.json: нет полей: id, name
2026-10-09 06:01:02,812 [WARNING] ingest: Пропущен not_object.json: ожидался объект, получено list
2026-10-09 06:01:02,815 [INFO] ingest: Отчёт сохранён: catalog.json



### Задание 17. CLI с кодами выхода

Точка входа `main()` — возвращает int: 0 при успехе, 1 при ошибке конвейера, 2 при ошибке конфигурации.

In [27]:
class IngestError(Exception):
    """Базовая ошибка импорта."""


def main() -> int:
    try:
        s = Settings.from_env()
    except ConfigError as exc:
        logger.error("Конфигурация невалидна: %s", exc)
        return 2

    try:
        rep = IngesterFull(s).run()
    except IngestError as exc:
        logger.error("Импорт прерван: %s", exc)
        return 1

    logger.info(
        "Готово: обработано %d из %d",
        rep["imported"],
        rep["total_files"],
    )
    return 0


print("Код выхода:", main())

WARNING ingest: Пропущен broken.json: битый JSON (Expecting value)
WARNING ingest: Пропущен no_keys.json: нет полей: id, name
WARNING ingest: Пропущен not_object.json: ожидался объект, получено list
INFO:ingest:Отчёт сохранён: catalog.json
INFO ingest: Отчёт сохранён: catalog.json
INFO:ingest:Готово: обработано 3 из 6
INFO ingest: Готово: обработано 3 из 6


Код выхода: 0


### Задание 18. Проверка сценариев

Запустите конвейер на подготовленных данных. Убедитесь, что валидные файлы импортированы, битые пропущены, код выхода 0.

In [28]:
code = main()
print("Код выхода:", code)

print("\ncatalog.json:")
print(open("catalog.json", encoding="utf-8").read())

WARNING ingest: Пропущен broken.json: битый JSON (Expecting value)
WARNING ingest: Пропущен no_keys.json: нет полей: id, name
WARNING ingest: Пропущен not_object.json: ожидался объект, получено list
INFO:ingest:Отчёт сохранён: catalog.json
INFO ingest: Отчёт сохранён: catalog.json
INFO:ingest:Готово: обработано 3 из 6
INFO ingest: Готово: обработано 3 из 6


Код выхода: 0

catalog.json:
{
  "source": "inbox",
  "total_files": 6,
  "imported": 3,
  "skipped": 3,
  "errors": [
    "broken.json: битый JSON (Expecting value)",
    "no_keys.json: нет полей: id, name",
    "not_object.json: ожидался объект, получено list"
  ],
  "records": [
    {
      "id": 1,
      "name": "Alice",
      "age": 30
    },
    {
      "id": 2,
      "name": "Bob",
      "age": 25
    },
    {
      "id": 3,
      "name": "Carol",
      "age": 28
    }
  ]
}


### Задание 19. Секреты и `.gitignore`

Создайте `.env` с настройками, загрузите через `python-dotenv`, убедитесь что `.env` в `.gitignore`.

In [29]:
%%writefile .env
SOURCE_DIR=inbox
OUTPUT_PATH=catalog.json
REQUIRED_KEYS=id,name

%%writefile .gitignore
.env
venv/
venv_*/
__pycache__/
*.pyc
*.log
.idea/
.vscode/
.ipynb_checkpoints/
inbox/
catalog.json

Writing .env


In [30]:
from dotenv import load_dotenv
import os

load_dotenv()
for key in ("SOURCE_DIR", "OUTPUT_PATH", "REQUIRED_KEYS"):
    print(f"{key} = {os.getenv(key)}")

!git status --short

WARNING dotenv.main: python-dotenv could not parse statement starting at line 4


SOURCE_DIR = inbox
OUTPUT_PATH = catalog.json
REQUIRED_KEYS = id,name
fatal: not a git repository (or any of the parent directories): .git


### Задание 20. Комплексный отчёт: единый pipeline

Соберите всё в единый модуль: `generate_report` + `main` + точка входа.

In [31]:
def generate_report(input_dir: str, output_path: str, required_keys: tuple) -> dict:
    """Читает каталог, валидирует документы, сохраняет отчёт."""
    os.environ["SOURCE_DIR"] = input_dir
    os.environ["OUTPUT_PATH"] = output_path
    os.environ["REQUIRED_KEYS"] = ",".join(required_keys)
    s = Settings.from_env()
    return IngesterFull(s).run()


def pipeline_main() -> int:
    """Оркестратор: настраивает логи, вызывает конвейер, печатает сводку."""
    setup_logging()
    try:
        rep = generate_report("inbox", "catalog.json", ("id", "name"))
    except ConfigError as exc:
        logger.error("Конфигурация: %s", exc)
        return 2

    print("\nСводка:")
    print(f"  Всего файлов: {rep['total_files']}")
    print(f"  Импортировано: {rep['imported']}")
    print(f"  Пропущено: {rep['skipped']}")
    for err in rep["errors"]:
        print(f"    - {err}")
    print(f"\nОтчёт: catalog.json")
    print(f"Лог: ingest.log")
    return 0


if __name__ == "__main__":
    raise SystemExit(pipeline_main())

WARNING ingest: Пропущен broken.json: битый JSON (Expecting value)
WARNING ingest: Пропущен broken.json: битый JSON (Expecting value)
WARNING ingest: Пропущен no_keys.json: нет полей: id, name
WARNING ingest: Пропущен no_keys.json: нет полей: id, name
WARNING ingest: Пропущен not_object.json: ожидался объект, получено list
WARNING ingest: Пропущен not_object.json: ожидался объект, получено list
INFO:ingest:Отчёт сохранён: catalog.json
INFO ingest: Отчёт сохранён: catalog.json
INFO ingest: Отчёт сохранён: catalog.json



Сводка:
  Всего файлов: 6
  Импортировано: 3
  Пропущено: 3
    - broken.json: битый JSON (Expecting value)
    - no_keys.json: нет полей: id, name
    - not_object.json: ожидался объект, получено list

Отчёт: catalog.json
Лог: ingest.log


SystemExit: 0

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


---
# Приложение. Шаблон кода

Скопируйте шаблон в отдельную ячейку или `.py`-файл и заполните `...`.

In [ ]:
"""
Лабораторная работа № 3.
Тема: Обработка исключений. Модуль OS. Виртуальные окружения.
Автор: <ФИО>
Дата: <дата>
"""
from __future__ import annotations

import json
import logging
import os
from logging.handlers import RotatingFileHandler
from pathlib import Path
from typing import Any, Iterator


# ============================================================
# 1. Исключения
# ============================================================
class IngestError(Exception):
    """Базовая ошибка конвейера импорта."""


class ConfigError(IngestError):
    """Конфигурация невалидна — работать нельзя."""


class DocumentError(IngestError):
    """Проблема с конкретным документом."""

    def __init__(self, path: Path, reason: str) -> None:
        super().__init__(f"{path.name}: {reason}")
        self.path = path
        self.reason = reason


# ============================================================
# 2. Настройки
# ============================================================
class Settings:
    def __init__(
        self,
        source_dir: Path,
        output_path: Path,
        required_keys: tuple[str, ...],
    ) -> None:
        self.source_dir = source_dir
        self.output_path = output_path
        self.required_keys = required_keys

    @classmethod
    def from_env(cls) -> "Settings":
        ...


# ============================================================
# 3. Логирование
# ============================================================
def setup_logging() -> None:
    ...


# ============================================================
# 4. Конвейер
# ============================================================
class Ingester:
    def __init__(self, settings: Settings) -> None:
        ...

    def iter_files(self) -> Iterator[Path]:
        ...

    def load_document(self, path: Path) -> dict[str, Any]:
        ...

    def run(self) -> dict[str, Any]:
        ...

    def _save(self, report: dict[str, Any]) -> None:
        ...


# ============================================================
# 5. CLI
# ============================================================
def main() -> int:
    ...


if __name__ == "__main__":
    raise SystemExit(main())

---
## Контрольные вопросы

1. Чем `else` в `try` отличается от кода в конце блока `try`?
2. Почему `finally` выполнится, даже если внутри `try` был `return`?
3. Почему `except:` без типа — плохо? Что он может поймать лишнего?
4. Что такое LBYL и EAFP? Приведите пример, где EAFP выигрывает.
5. Зачем наследовать собственное исключение от `Exception`, а не от `BaseException`?
6. Что произойдёт, если открыть файл в режиме `'w'` и он существует?
7. Зачем в `open` указывать `encoding`?
8. Чем `pathlib.Path` лучше `os.path`? Есть ли случаи, где `os.path` предпочтительнее?
9. Зачем нужен `if __name__ == "__main__"`?
10. Что такое `__init__.py` и почему без него пакет может работать не везде?
11. Зачем виртуальное окружение? Что произойдёт без него через полгода разработки?
12. Почему `.env` нельзя коммитить? Что делать, если он уже попал в репозиторий?
13. Зачем логирование, если есть `print`? Приведите три причины.
14. Что такое ротация логов и зачем она нужна?
15. Как `logging.getLogger(__name__)` помогает в больших проектах?

---
## Дополнительно

1. Оформите ноутбук как `lab3_exceptions_files.ipynb`.
2. Убедитесь, что он выполняется сверху вниз без ошибок (Kernel → Restart & Run All).
3. Создайте в GitLab репозиторий `python-labs` и папку `lab3`.
4. Добавьте файлы: ноутбук, `catalog.json`, `ingest.log`, `README.md`, `.gitignore`.
5. Закоммитьте и запушьте:

```bash
git add .
git commit -m "ЛР3: исключения, файлы, логирование, окружение"
git push origin main
```

6. В конце README укажите ссылку на репозиторий.
7. Прикрепите ссылку и ноутбук в системе дистанционного обучения.